<a href="https://colab.research.google.com/github/bradleywjenks/wds-modelling-optimization-course/blob/main/notebooks/model_calibration_coursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hydraulic model calibration coursework

## Project brief
Anglian Water (AW) requires an evaluation and calibration of the hydraulic model of **EXNING**, a district metered area (DMA) in its water distribution network. AW has provided:
- a hydraulic model of the EXNING DMA, previously calibrated in 2019,
- loading conditions (demands, reservoir heads) and head measurements at 14 sensors covering a period of 4 days,
- pipe groups with bounds on each group's C-factor.

The deliverable is expected to be a short calibration report for AW. This will include code completed in this notebook, answers to the questions in each part of the calibration, and a set of recommendations to AW.

### Workplan
1. Preliminary model evaluation
2. Part 1: estimate pipe C-factors **without** pipe grouping
3. Part 2: estimate pipe C-factors **with** pipe grouping
4. Part 3: estimate pipe C-factors **with** pipe grouping **and** valve loss coefficients
5. Discussion and recommendations to AW

## Setup
Run the cells below to set up the notebook environment.
- **Google Colab:** the first cell clones (or updates) the repository. In the first instance, it also installs the `opwater` helper package and its dependencies, then restarts the session (Colab reports that the session crashed; this is expected). **Once it reconnects, run the first cell again**, then continue from the second cell. A pip warning about `numba` and `numpy` can be ignored.
- **Local:** install the repository first (see the README); the first cell is then skipped.

The second cell imports the packages used in this notebook.

In [ ]:
import sys

if "google.colab" in sys.modules:
    import importlib.util
    import os

    repo_dir = "/content/wds-modelling-optimization-course"
    if os.path.isdir(repo_dir):
        !git -C {repo_dir} pull --quiet --ff-only
    else:
        !git clone --depth 1 https://github.com/bradleywjenks/wds-modelling-optimization-course.git {repo_dir}

    # install opwater from the clone, unless this runtime already has it
    spec = importlib.util.find_spec("opwater")
    if spec is None or not spec.origin.startswith(repo_dir):
        %pip install --quiet -e {repo_dir}
        print("Restarting the session to load the new packages. Run this cell again once it reconnects.")
        os.kill(os.getpid(), 9)

In [ ]:
import copy
import warnings

import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib_inline
import networkx as nx
import numpy as np
import wntr
from numpy import linalg as la

from opwater import data_dir, load_network_data, network_file

warnings.filterwarnings("ignore")

# Matplotlib parameters
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

plt.rc("font", size=10)
plt.rc("axes", titlesize=10)
plt.rc("axes", labelsize=11)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)
plt.rc("legend", fontsize=10)

## Load network properties and operational data

Load the network .inp file and operational data from the repository's `data` folder.

In [ ]:
net_name = 'exning.inp'
data_name = 'exning_coursework_dataset.npy'
inp_file = network_file(net_name)

# load operational data
data = np.load(data_dir() / 'operational' / data_name, allow_pickle=True).item()
h_data = data['h_data']
sensor_idx = data['sensor_idx'] - 1 # matlab to python index offset
d_data = data['d']
h0_data = data['h0'].reshape(-1, 1).T

# load network properties
wdn = load_network_data(inp_file)
A12 = wdn.A12
A10 = wdn.A10
net_info = wdn.net_info
link_df = wdn.link_df
node_df = wdn.node_df
demand_df = wdn.demand_df

Define a function to plot the network and sensor locations to be used throughout the notebook.

In [ ]:
# define plotting function using networkx
def plot_network(wdn, sensor_idx, vals=None, highlight_valves=None):

    # unload data
    link_df = wdn.link_df
    node_df = wdn.node_df
    net_info = wdn.net_info
    h0_df = wdn.h0_df

    # draw network
    uG = nx.from_pandas_edgelist(link_df, source='node_out', target='node_in')
    pos = {row['node_ID']: (row['xcoord'], row['ycoord']) for _, row in node_df.iterrows()}
    nx.draw(uG, pos, node_size=20, node_shape='o', node_color='gray')

    # draw reservoir
    nx.draw_networkx_nodes(uG, pos, nodelist=net_info['reservoir_names'], node_size=100, node_shape='s', node_color='black')

    # draw sensor nodes
    sensor_names = [net_info['junction_names'][i] for i in sensor_idx]
    nx.draw_networkx_nodes(uG, pos, sensor_names, node_size=100, node_shape='o', node_color='red', edgecolors='white')

    # reservoir labels
    reservoir_labels = {node: 'Reservoir' for node in net_info['reservoir_names']}
    labels_res = nx.draw_networkx_labels(uG, pos, reservoir_labels, font_size=12, verticalalignment='top')
    for _, label in labels_res.items():
        label.set_y(label.get_position()[1] - 1500)

    # sensor labels
    sensor_labels = {node: str(idx+1) for (idx, node) in enumerate(sensor_names)}
    labels_sen = nx.draw_networkx_labels(uG, pos, sensor_labels, font_size=12, verticalalignment='bottom')
    for _, label in labels_sen.items():
        label.set_y(label.get_position()[1] + 1000)
        
    # plot sensor vals
    if vals is not None:
        
        # diverging colour map centred on zero; limits set by the largest absolute residual
        vmax = np.max(np.abs(vals))
        
        # plot residuals
        nodes = nx.draw_networkx_nodes(uG, pos, nodelist=sensor_names, node_size=100, node_shape='o', node_color=vals, cmap='RdBu_r', vmin=-vmax, vmax=vmax, edgecolors='black', linewidths=0.5)
        
        # create color bar
        colorbar = plt.colorbar(nodes, ax=plt.gca())
        colorbar.set_label('Mean pressure residual [m]', fontsize=12)
        
    # highlight link
    if highlight_valves is not None:
        nx.draw_networkx_nodes(uG, pos, highlight_valves, node_size=200, node_shape='d', node_color='limegreen', edgecolors='white')

In [ ]:
# TODO: plot the network and sensor locations
plot_network(wdn, ...)

## Preliminary model evaluation

Simulate network hydraulics of the initial (uncalibrated) model over the 4-day period. The function below solves the network hydraulics using the `wntr` package, as done so in the hydraulic modelling notebook. Specifically, it:
- loads the network properties from the `inp_file`
- sets the simulation duration to match the operational data
- assigns the measured heads `h0_data` to the reservoir
- assigns the demand data `d_data` to each junction
- includes an optional argument `theta` to input pipe C-factors and valve loss coefficients (stacked vertically); `theta=None` is the default which assigns the original parameters from the .inp file
- includes an optional argument `demand` to return the set of model demands only if True

In [ ]:
#### DO NOT CHANGE THIS ####
def hydraulic_solver(inp_file, d_data, h0_data, theta=None, demand=False):

    # load network from wntr
    wn = wntr.network.WaterNetworkModel(str(inp_file))

    # get network properties
    reservoir_names = wn.reservoir_name_list
    junction_names = wn.junction_name_list
    link_names = wn.link_name_list

    # modify simulation time and hydraulic time step
    nt = h0_data.shape[1]
    wn.options.time.duration = (nt - 1) * 3600
    wn.options.time.hydraulic_timestep = 3600
    wn.options.time.pattern_timestep = 3600
    wn.options.time.report_timestep = 3600

    # assign reservoir data
    for (i, name) in enumerate(reservoir_names):
        wn.add_pattern(f'h0_{i}', h0_data[i])
        reservoir = wn.get_node(name)
        reservoir.head_timeseries.base_value = 1
        reservoir.head_timeseries.pattern_name = wn.get_pattern(f'h0_{i}')

    # replace demand data
    for idx, name in enumerate(junction_names):

        if any(val != 0 for val in d_data[idx, :]):
            node = wn.get_node(name)
            d_pat = d_data[idx, :]
            wn.add_pattern('d_'+name, d_pat)

            for (i, num) in enumerate(node.demand_timeseries_list):
                if i == 0:
                    node.demand_timeseries_list[i].base_value = 1
                    node.demand_timeseries_list[i].pattern_name = 'd_'+name
                else: 
                    node.demand_timeseries_list[i].base_value = None
                    node.demand_timeseries_list[i].pattern_name = None

    # assign C-factors
    if theta is not None:
        for name, link in wn.links():

            # check if the link is a pipe
            if isinstance(link, wntr.network.Pipe):
                link.roughness = theta[link_names.index(name)]

            # check if link is a valve
            elif isinstance(link, wntr.network.Valve):
                link.minor_loss = theta[link_names.index(name)]
                link.initial_setting = theta[link_names.index(name)]

    # run simulation and get results
    sim = wntr.sim.EpanetSimulator(wn)
    results = sim.run_sim()

    q_sim = results.link['flowrate'].T
    h_sim = results.node['head'].T
    h_sim = h_sim[~h_sim.index.isin(reservoir_names)] # delete reservoir nodes
    d = results.node['demand'].T
    d = d[~d.index.isin(reservoir_names)] # delete reservoir nodes


    if demand == True:
        return d.to_numpy()
    else:
        return q_sim.to_numpy(), h_sim.to_numpy()

Run the simulation with the initial coefficients `theta_0`.

In [ ]:
theta_0 = link_df['C'].to_numpy() # NB: C-factors and valve coefficients labelled as `C` in .inp file
# TODO: simulate initial model hydraulics over the 4-day period
_, h_0 = hydraulic_solver(...)

Compare the simulated and measured heads at sensor nodes (node indices in `sensor_idx`) over the 4-day period. Visualize the results with, for example, a boxplot of pressure residuals and/or a plot of the network with mean pressure residuals at the sensor nodes. Other graphics or tables are encouraged so long as they support the discussion in Question 1; unnecessary reporting will be penalized.

In [ ]:
# TODO: compute and visualize the pressure residuals at the sensor nodes
# compute pressure residuals at the sensor nodes
residuals_0 = ...

# boxplot of the pressure residuals at each sensor
fig, ax = plt.subplots()
...
ax.set_xlabel('Sensor index', fontsize=12)
ax.set_ylabel('Pressure residual [m]', fontsize=12)

# Consider other results that support Question 1, for example:
# - the spatial distribution of the mean residuals (plot_network accepts vals=...)
# - the residuals over time (are they flow dependent?)
# - the initial C-factors theta_0

**<u>Question 1:</u>** Comment on the results of the preliminary model evaluation. In particular, explain the magnitude of the pressure residuals at the sensors. Is the current hydraulic model of EXNING accurate according to hydraulic model calibration guidelines? What are the most likely sources of model error?

<font color="red">Enter response here...</font>

## Part 1: parameter estimation without pipe grouping (valves fixed)

Following initial reports of discrepancies in the EXNING model, AW confirmed that:
- the flow and head sensors were calibrated before the data were collected, and
- the demands in the dataset are accurate (including large users).

The remaining differences between simulated and measured heads must therefore result from inaccurate model parameters. Calibration is posed as a parameter estimation problem; that is, find the parameters that minimize the discrepancy between simulated and measured heads. In Part 1, pipe C-factors are estimated without grouping and valve status is assumed to be known with loss coefficients fixed at $\theta = 0.2$ (i.e., fully open).

Split the operational data into training and test datasets.

In [ ]:
# train data
nt_train = ...  # TODO: number of hourly time steps used for training (the dataset has 96)
data_train = {
    'd': d_data[:, :nt_train],
    'h0': h0_data[:, :nt_train],
    'h_data': h_data[:, :nt_train]
}

# test data
data_test = {
    'd': d_data[:, nt_train:],
    'h0': h0_data[:, nt_train:],
    'h_data': h_data[:, nt_train:]
}

Define the loss function.

In [ ]:
def loss_fun(h, h_data):
    # TODO: define loss function
    return ...

Compute the MSE of the initial model (`theta_0`) on the training data.

In [ ]:
mse_0 = ...  # TODO: MSE of the initial model over the training period (first nt_train time steps of h_0)

**<u>Question 2:</u>** Briefly describe the hydraulic model calibration problem. Justify the choice of training dataset and the definition of `loss_fun`.

<font color="red">Enter response here...</font>

Solve the parameter estimation problem **without** pipe grouping. The function below computes the coefficients of the linear approximation of the head loss equations at the current flow and $\theta$ estimates.

In [ ]:
# compute matrices for the head loss linear approximations
def linear_approx_calibration(wdn, q, theta):

    # unload data
    A12 = wdn.A12
    A10 = wdn.A10
    net_info = wdn.net_info
    link_df = wdn.link_df

    K = np.zeros((net_info['np'], 1))
    n_exp = link_df['n_exp'].astype(float).to_numpy().reshape(-1, 1)
    b1_k = copy.copy(K)
    b2_k = copy.copy(K)

    for idx, row in link_df.iterrows():
        if row['link_type'] == 'pipe':
            K[idx] = 10.67 * row['length'] * (theta[idx] ** -row['n_exp']) * (row['diameter'] ** -4.8704)
            b1_k[idx] = copy.copy(K[idx])
            b2_k[idx] = (-n_exp[idx] * K[idx]) / theta[idx]

        elif row['link_type'] == 'valve':
            K[idx] = (8 / (np.pi ** 2 * 9.81)) * (row['diameter'] ** -4) * theta[idx]
            b1_k[idx] = -n_exp[idx] * copy.copy(K[idx]) 
            b2_k[idx] = copy.copy(K[idx]) / theta[idx]

    a11_k = np.tile(K, q.shape[1]) * np.abs(q) ** (n_exp - 1)
    b1_k = np.tile(b1_k, q.shape[1]) * np.abs(q) ** (n_exp - 1)
    b2_k = np.tile(b2_k, q.shape[1]) * np.abs(q) ** (n_exp - 1) * q

    return a11_k, b1_k, b2_k

Implement SCP with `cvxpy`.

**A note on flow units:** the flow variable `q` in the convex subproblem is converted to L/s to avoid numerical scaling issues. However, the hydraulic solver in `wntr` accepts demands `d` and outputs flow in m³/s. Expressing `q` in L/s changes two constraint terms in the optimization problem:
1. Energy conservation: $n\,a_{11}\,q = n\,(a_{11}/1000)\,q$, so the coefficient on `q` is divided by 1000. The other terms (`b1_k * q_k` and `b2_k * theta`) do not contain `q` and are unchanged.
2. Mass conservation: $A_{12}^\top q = 1000\,d$, where `d` is converted to L/s by multiplying by 1000.

The objective and the trust region depend only on `h` and `theta`, so the solution is unchanged. The SCP updates `q_k` with the hydraulic solver in units of m³/s.

In [ ]:
# TODO: complete the lines marked ... to implement SCP

# unload network training data
n_exp = ...
d = ...
h_data = ...
h0 = ...

# define problem parameters
Ki = np.inf 
iter_max = 50
delta_k = 20
theta_up_pipe = 200 # upper bound on theta for pipe links
theta_lo_pipe = 25 # lower bound on theta for pipe links

# initialize values
theta_k = ...
q_k, h_k = hydraulic_solver(...)
a11_k, b1_k, b2_k = linear_approx_calibration(...)
objval_k = loss_fun(...)

### main scp code ###
print(f"{'k':>3} {'objval_k':>10} {'Ki':>10} {'delta_k':>9}  step")
for k in range(iter_max):

    # decision variables
    q = ...
    h = ...
    theta = ...

    # objective function (defined using CVXPY functions)
    loss = ...  # use cp.sum_squares
    objective = cp.Minimize(loss)

    # hydraulic feasibility constraints
    constraints = []
    for t in range(nt_train):
        # energy and mass constraints (q in L/s for numerical conditioning; q_k, a11_k and d are in m^3/s)
        constraints += [
            cp.multiply(b1_k[:, t], q_k[:, t]) + cp.multiply(cp.multiply(n_exp, a11_k[:, t] / 1000), q[:, t]) + cp.multiply(b2_k[:, t], theta) + A12 @ h[:, t] + A10 @ h0[:, t] == 0,
            ...  # mass conservation at time t (q is in L/s, d is in m³/s)
        ]

    # trust region constraints
    constraints += [
        ...  # limit the change in theta from theta_k to delta_k (hint: cp.norm with the infinity norm)
    ]

    # variable bounds
    # bound the pipe C-factors between theta_lo_pipe and theta_up_pipe (link types are in link_df['link_type'])
    ...

    # valve local loss coefficient constraints
    # fix the valve loss coefficients at their initial values in theta_0
    ...

    # solve optimization problem
    problem = cp.Problem(objective, constraints)
    cvx_val = problem.solve(solver=cp.CLARABEL, verbose=False)

    # store optimal solution and compute actual and predicted decrease in objval
    theta_tilde = theta.value
    [q_tilde, h_tilde] = hydraulic_solver(...)
    objval_tilde = loss_fun(...)

    predicted_decrease = ...
    actual_decrease = ...

    # stop if the linearized model predicts a negligible decrease
    if predicted_decrease <= 1e-3 * np.abs(objval_k):
        print(f"{k:>3} {objval_k:>10.4f} {Ki:>10.2e} {delta_k:>9.3f}  converged: negligible predicted decrease")
        break

    # evaluate latest SCP iteration
    if ...:  # accept the step if the ratio of actual to predicted decrease is at least 0.1
        objval_old = objval_k
        # update theta_k, q_k, h_k and objval_k, then recompute the linear approximation at the new estimate
        ...
        Ki = np.abs(objval_old - objval_k) / np.abs(objval_old)
        delta_k = 1.1 * delta_k
        step = "successful: update estimate, increase trust region size"

    else:
        delta_k = 0.25 * delta_k
        step = "unsuccessful: keep estimate, decrease trust region size"

    print(f"{k:>3} {objval_k:>10.4f} {Ki:>10.2e} {delta_k:>9.3f}  {step}")

    if Ki <= 1e-3 or np.abs(objval_k) <= 1e-2 or delta_k <= 1e-1:
        break

**<u>Question 3:</u>** Describe the role of each value printed by the SCP algorithm above (`objval_k`, `Ki` and `delta_k`) and explain how each changes over the iterations.

<font color="red">Enter response here...</font>

In [ ]:
theta_1 = theta_k

# compute train mse
mse_train_1 = ...  # TODO

Evaluate the Part 1 calibration results on the test dataset.

In [ ]:
# TODO: simulate the test period with theta_1 and compute the test MSE
_, h_test = hydraulic_solver(...)
mse_test_1 = ...

In [ ]:
# TODO: compute the test pressure residuals and plot them as in the preliminary evaluation
residuals_1 = ...

**<u>Question 4:</u>** Briefly comment on the change in model accuracy before and after calibration. According to hydraulic model calibration guidelines, can the new model be considered calibrated? Compare model calibration guidelines in the US (AWWA) and UK (Water Research Centre).

<font color="red">Enter response here...</font>

In [ ]:
# calibrated pipe C-factors (theta_1 also holds the valve loss coefficients)
pipe_theta = [theta_1[idx] for idx, row in link_df.iterrows() if row['link_type'] == 'pipe']

# TODO: plot pipe_theta against pipe index; consider comparing with theta_0 and the bounds on theta

**<u>Question 5:</u>** Comment on the results of the model calibration without pipe grouping. In particular, discuss the calibrated `theta_1` values. Are they realistic? Why or why not?

<font color="red">Enter response here...</font>

## Part 2: parameter estimation with pipe grouping (valves fixed)

AW has requested that the calibration be repeated with pipe grouping. Pipes in the same group share one C-factor. Each group's bounds are $[\theta_0 - 25, \theta_0]$, where $\theta_0$ is the group's C-factor in the .inp file, so calibration can only lower the C-factors (e.g., due to pipe aging), by at most 25. Load the pipe grouping data.

In [ ]:
link_groups = data['G'][0] - 1 # matlab to python index offset
ng = link_groups.shape[0]
thetaG_lo = data['thetaG_min'][:, 0]
thetaG_up = data['thetaG_max'][:, 0]

for j in range(ng):
    print(f"Group {j + 1}: {len(link_groups[j])} pipes, theta_0 = {theta_0[link_groups[j][0, 0]]:.0f}, bounds = [{thetaG_lo[j]}, {thetaG_up[j]}]")

Copy the SCP implementation from Part 1 and add the pipe grouping constraints. As in Part 1, the flow variable `q` is in L/s.

In [ ]:
# TODO: SCP with pipe grouping, starting from the Part 1 implementation. Add a decision variable for the
# C-factor of each group (e.g., thetaG = cp.Variable(ng)) and constraints so that pipes in group j
# (link_groups[j]) share the C-factor thetaG[j], bounded by thetaG_lo[j] and thetaG_up[j]
...

In [ ]:
theta_2 = theta_k

# compute train mse
mse_train_2 = ...  # TODO

Evaluate the results on the test dataset.

In [ ]:
# TODO: simulate the test period with theta_2 and compute the test MSE
mse_test_2 = ...

In [ ]:
# TODO: compute and plot the test pressure residuals
residuals_2 = ...

**<u>Question 6:</u>** Comment on the difference in model accuracy after calibration with (`theta_2`) and without (`theta_1`) pipe grouping. According to hydraulic model calibration guidelines, can the model with `theta_2` be considered calibrated?

<font color="red">Enter response here...</font>

In [ ]:
# TODO: plot the calibrated pipe C-factors theta_2

**<u>Question 7:</u>** Compare the H-W coefficient estimates in `theta_2` with those in `theta_1`. Explain any differences.

<font color="red">Enter response here...</font>

### Identify sensors with the largest residuals

In [ ]:
# TODO: plot the mean test pressure residuals at the sensor nodes
mean_residuals_2 = ...
plot_network(...)

**<u>Question 8:</u>** Interpret the differences between simulated and measured pressure heads by answering the following questions:
* Are the pressure residuals centered on zero?
* Does the hydraulic model overestimate or underestimate pressure heads?
* What could be the nature and location of the fault causing these residuals?

<font color="red">Enter response here...</font>

## Part 3: parameter estimation with pipe grouping (valve coefficients free)

Localized head losses may result from unrecorded changes in the network (faults), such as a partially closed valve. Faults of this kind may be identified by solving the parameter estimation problem with the valve loss coefficients free to vary.

In Part 3, propose and investigate an approach to the fault localization problem. Suggested steps are as follows:
1. Modify the SCP implementation in Part 2 so the valve loss coefficients can vary (Part 3a). Solve the resulting parameter estimation problem and interpret the estimates `theta_3a`.
2. Consider the expected values of the valve loss coefficients in the fault localization problem. Add a regularization term to the objective function that reflects this and solve the problem again (Part 3b). Interpret the estimates `theta_3b`.
3. Compare the results of Part 3 with Part 2. Explain whether they support or contradict the earlier conclusions (Question 8) about the nature and location of the fault.

### Part 3a: free valve loss coefficients

Modify the SCP implementation in Part 2 so the valve loss coefficients are free to vary, with suitable bounds. The valve loss coefficients must remain positive; $\theta = 0.2$ corresponds to a fully open valve. As in Part 1, the flow variable `q` is in L/s.

In [ ]:
# TODO: SCP with pipe grouping and free valve loss coefficients
...

In [ ]:
theta_3a = theta_k

# compute train mse
mse_train_3a = ...  # TODO

Evaluate the results on the test dataset.

In [ ]:
# TODO: evaluate theta_3a on the test dataset (test MSE and pressure residuals), then plot the
# estimated pipe C-factors, valve loss coefficients and mean pressure residuals at the sensors

**<u>Question 9:</u>** Comment on the model accuracy and the parameter estimates in `theta_3a`. Can the fault be located from these results?

<font color="red">Enter response here...</font>

### Part 3b: free valve loss coefficients with regularization
Regularization adds a penalty term to the objective function that encodes prior knowledge about the parameters. Here, the penalty acts on the deviations of the parameter estimates from their prior values, $\theta - \theta_0$, and is weighted by a hyperparameter $\lambda \geq 0$ that controls the trade-off between fitting the data and staying close to the prior:
- **Ridge (L2 or Tikhonov) regularization** adds $\lambda\,\|\theta - \theta_0\|_2^2$. Shrinks parameters smoothly toward zero (never exactly zero), keeping all features in the model.
- **Lasso (L1) regularization** adds $\lambda\,\|\theta - \theta_0\|_1$. Drives parameters completely to zero, removing irrelevant features (i.e., feature selection).


**<u>Question 10:</u>** Which regularization function should be added to the objective function of the fault localization problem? Justify the choice, including the parameters it applies to and the range of hyperparameters tested.

<font color="red">Enter response here...</font>

Define the regularization term.

In [ ]:
# TODO: define the regularization term and its hyperparameter
# (a NumPy version is useful for evaluating the objective value and a CVXPY version for the optimization problem)
...

Run SCP with free valve loss coefficients and regularization. As in Part 1, the flow variable `q` is in L/s.

In [ ]:
# TODO: SCP with pipe grouping, free valve loss coefficients and regularization
# (include the regularization term in objval_k and objval_tilde so the predicted and actual decreases refer to the same objective)
...

In [ ]:
theta_3b = theta_k

# compute train mse
mse_train_3b = ...  # TODO

Evaluate the results on the test dataset.

In [ ]:
# TODO: evaluate theta_3b on the test dataset (test MSE and pressure residuals), then plot the
# estimated pipe C-factors, valve loss coefficients and mean pressure residuals at the sensors

Identify the faulty valves and plot their locations. `plot_network` marks a list of node names given by its `highlight_valves` argument (e.g., the start node of each faulty valve, `link_df['node_out']`).

In [ ]:
# TODO: identify the faulty valves and plot their locations
...

**<u>Question 11:</u>** Comment on the results of Part 3b and compare them with Parts 2 and 3a. Do they support or contradict the earlier conclusions (Question 8) about the nature and location of the fault? How does the solution change with different regularization hyperparameters?

<font color="red">Enter response here...</font>

## Discussion

**<u>Question 12:</u>** Summarize the findings for AW. What are the most likely source(s) of error in the EXNING model and what model update is proposed? Recommend how AW should validate the proposed model update.

<font color="red">Enter response here...</font>

**<u>Question 13:</u>** What might be different about calibrating a hydraulic model of a typical North American water distribution system compared with the EXNING DMA? What additional field work might be required for model calibration and why?

Provide a brief step-by-step guideline for calibrating a hydraulic model of a pressure district in the City of Toronto. What information would be needed and why?

<font color="red">Enter response here...</font>